In [1]:
import torch
print(torch.backends.mps.is_available())

True


In [2]:
import sqlite3
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import os

# 1. Load the clinical dataset
data = load_breast_cancer(as_frame=True)
df = data.frame

# 2. Shuffle and partition into 3 distinct hospitals
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
partitions = {
    "hospital_A": df.iloc[:190],
    "hospital_B": df.iloc[190:380],
    "hospital_C": df.iloc[380:]
}

# 3. Create isolated SQLite databases for each hospital node
os.makedirs("privacare_dbs", exist_ok=True)

for hospital_name, hospital_df in partitions.items():
    # Enforce 80/20 Train/Test Split locally
    train_df, test_df = train_test_split(hospital_df, test_size=0.2, random_state=42)
    
    # Connect to the isolated SQLite database
    db_path = f"privacare_dbs/{hospital_name}.db"
    conn = sqlite3.connect(db_path)
    
    # Write isolated tables
    train_df.to_sql("train_data", conn, if_exists="replace", index=False)
    test_df.to_sql("test_data", conn, if_exists="replace", index=False)
    
    conn.close()
    print(f"Provisioned {hospital_name}.db | Train records: {len(train_df)} | Test records: {len(test_df)}")

Provisioned hospital_A.db | Train records: 152 | Test records: 38
Provisioned hospital_B.db | Train records: 152 | Test records: 38
Provisioned hospital_C.db | Train records: 151 | Test records: 38


In [3]:
import torch
import torch.nn as nn

# Select Apple Silicon acceleration (MPS) if available, otherwise fall back to CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

class DiagnosticNet(nn.Module):
    def __init__(self):
        super(DiagnosticNet, self).__init__()
        # Input layer: 30 clinical features -> 16 hidden neurons
        self.layer1 = nn.Linear(30, 16)
        self.relu = nn.ReLU()
        # Second layer: 16 neurons -> 8 hidden neurons
        self.layer2 = nn.Linear(16, 8)
        # Output layer: 8 neurons -> 1 single output (0 or 1)
        self.output = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.sigmoid(self.output(x))
        return x

# Instantiate the network and push it to device
model = DiagnosticNet().to(device)
print(f"Model initialized on: {device}")
print(model)

Model initialized on: mps
DiagnosticNet(
  (layer1): Linear(in_features=30, out_features=16, bias=True)
  (relu): ReLU()
  (layer2): Linear(in_features=16, out_features=8, bias=True)
  (output): Linear(in_features=8, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [4]:
import sqlite3
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Connect to Hospital A's isolated SQLite database
conn = sqlite3.connect("privacare_dbs/hospital_A.db")
train_df = pd.read_sql("SELECT * FROM train_data", conn)
conn.close()

# 2. Extract clinical features (X) and diagnosis target (y)
X = train_df.drop(columns=["target"]).values
y = train_df["target"].values

# 3. Convert to PyTorch tensors and move to hardware accelerator (MPS/CPU)
X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1).to(device)

# 4. Setup loss function and optimizer
criterion = nn.BCELoss() # Binary Cross Entropy for 0/1 classification
optimizer = optim.Adam(model.parameters(), lr=0.005)

# 5. Execute local training loop
epochs = 10
print("Starting local training at Hospital A...")
model.train()

for epoch in range(epochs):
    optimizer.zero_grad()          # Clear old gradients
    outputs = model(X_tensor)      # Forward pass
    loss = criterion(outputs, y_tensor) # Calculate error
    loss.backward()                # Backpropagation
    optimizer.step()               # Update weights

    if (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

Starting local training at Hospital A...
Epoch 2/10 | Loss: 2.4297
Epoch 4/10 | Loss: 0.6365
Epoch 6/10 | Loss: 0.6784
Epoch 8/10 | Loss: 0.6134
Epoch 10/10 | Loss: 0.6005


In [5]:
import torch
import copy

def apply_differential_privacy(local_model, noise_multiplier=0.05):
    """
    Injects Gaussian noise into the model's weights to enforce Differential Privacy.
    """
    # Create a copy so we don't destroy the original training weights
    dp_model = copy.deepcopy(local_model)
    
    with torch.no_grad():
        for param in dp_model.parameters():
            # Generate Gaussian noise matching the parameter's shape and device
            noise = torch.normal(mean=0.0, std=noise_multiplier, size=param.size()).to(device)
            # Inject noise directly into the weights
            param.add_(noise)
            
    return dp_model

# Let's test the privacy shield on our trained model from Hospital A
print("Applying Differential Privacy...")

# 1. Capture a sample weight BEFORE noise
original_weight = model.layer1.weight[0][0].item()
print(f"Original Weight (Sample): {original_weight:.6f}")

# 2. Apply the Gaussian noise
dp_secured_model = apply_differential_privacy(model, noise_multiplier=0.05)

# 3. Capture the same weight AFTER noise
noised_weight = dp_secured_model.layer1.weight[0][0].item()
print(f"Noised Weight (Sample):   {noised_weight:.6f}")

Applying Differential Privacy...
Original Weight (Sample): -0.033028
Noised Weight (Sample):   0.025921


In [1]:
import flwr as fl

# Define the Federated Averaging (FedAvg) strategy
strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0,        # Ask 100% of available clients to train
    fraction_evaluate=1.0,   # Ask 100% of available clients to evaluate
    min_fit_clients=3,       # Wait for all 3 hospitals (A, B, C) to connect
    min_evaluate_clients=3,  # Wait for all 3 hospitals to evaluate
    min_available_clients=3, # Require 3 hospitals to be online before starting
)

print("Starting PrivaCare Central Orchestrator...")

# Construct the Flower central server orchestration loop running the FedAvg aggregation algorithm
fl.server.start_server(
    server_address="127.0.0.1:8080",
    config=fl.server.ServerConfig(num_rounds=3), # We will run 3 full global learning rounds
    strategy=strategy,
)

	Instead, use the `flower-superlink` CLI command to start a SuperLink as shown below:

		$ flower-superlink --insecure

	To view usage and all available options, run:

		$ flower-superlink --help

	Using `start_server()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower server, config: num_rounds=3, no round_timeout
INFO :      Flower ECE: gRPC server running (3 rounds), SSL is disabled
INFO :      [INIT]
INFO :      Requesting initial parameters from one random client


Starting PrivaCare Central Orchestrator...


I1003 23:02:22.719279  329579 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
INFO :      Received initial parameters from one random client
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 3 clients (out of 3)
INFO :      aggregate_fit: received 3 results and 0 failures
INFO :      configure_evaluate: strategy sampled 3 clients (out of 3)
INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [ROUND 2]
INFO :      configure_fit: strategy sampled 3 clients (out of 3)
INFO :      aggregate_fit: received 3 results and 0 failures
INFO :      configure_evaluate: strategy sampled 3 clients (out of 3)
INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [ROUND 3]
INFO :      configure_fit: strategy sampled 3 clients (out of 3)
INFO :    

History (loss, distributed):
	round 1: 0.8793507615725199
	round 2: 24.714697519938152
	round 3: 0.7787169814109802

In [1]:
import pandas as pd
import sqlite3
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Load the Kaggle dataset from your folder
df = pd.read_csv("diabetes_prediction_dataset.csv")

# 2. Encode text columns (gender, smoking_history) into numbers for PyTorch
le = LabelEncoder()
df['gender'] = le.fit_transform(df['gender'])
df['smoking_history'] = le.fit_transform(df['smoking_history'])

# Rename the 'diabetes' column to 'target' so it matches your existing client.py logic
df = df.rename(columns={'diabetes': 'target'})

# 3. Shuffle and partition into 3 distinct hospitals
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
partition_size = len(df) // 3

partitions = {
    "hospital_A": df.iloc[:partition_size],
    "hospital_B": df.iloc[partition_size:partition_size*2],
    "hospital_C": df.iloc[partition_size*2:]
}

# 4. Overwrite your existing SQLite databases with the new massive dataset
os.makedirs("privacare_dbs", exist_ok=True)

for hospital_name, hospital_df in partitions.items():
    # Enforce 80/20 Train/Test Split locally
    train_df, test_df = train_test_split(hospital_df, test_size=0.2, random_state=42)
    
    # Connect to the isolated SQLite database
    db_path = f"privacare_dbs/{hospital_name}.db"
    conn = sqlite3.connect(db_path)
    
    # Write isolated tables (this replaces the old breast cancer data)
    train_df.to_sql("train_data", conn, if_exists="replace", index=False)
    test_df.to_sql("test_data", conn, if_exists="replace", index=False)
    
    conn.close()
    print(f"Provisioned {hospital_name}.db | Train records: {len(train_df)} | Test records: {len(test_df)}")

Provisioned hospital_A.db | Train records: 26666 | Test records: 6667
Provisioned hospital_B.db | Train records: 26666 | Test records: 6667
Provisioned hospital_C.db | Train records: 26667 | Test records: 6667
